## hmmmm

In [2]:
import re
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import pymupdf

In [3]:
df = pd.read_csv("glints_jobs_detail.csv")

In [4]:
df.columns.tolist()

['Judul',
 'Industri',
 'Tipe',
 'Kategori',
 'Tanggal_Post',
 'Berlaku_Hingga',
 'Perusahaan',
 'Website_Perusahaan',
 'Logo_Perusahaan',
 'Alamat',
 'Kota',
 'Provinsi',
 'Negara',
 'Latitude',
 'Longitude',
 'Gaji_Min',
 'Gaji_Max',
 'Gaji_Currency',
 'Gaji_Unit',
 'Skills',
 'Pengalaman_Bulan',
 'Pendidikan',
 'Deskripsi',
 'Tentang_Perusahaan',
 'Benefit',
 'Apply_Langsung',
 'Link']

In [5]:
job_texts = df['Skills'].astype(str)

In [6]:

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#[A-Za-z0-9]+', '', text)
    text = re.sub(r'RT[\s]', '', text)
    text = re.sub(r"http\S+", '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)

    text = text.replace('\n', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip(' ')
    return text

def casefoldingText(text):
    text = text.lower()
    return text

def tokenizingText(text):
    text = word_tokenize(text)
    return text

def filteringText(text):
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text

def stemmingText(text):
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    words = text.split()

    stemmed_words = [stemmer.stem(word) for word in words]
    stemmed_text = ' '.join(stemmed_words)

    return stemmed_text

def toSentence(list_words):
    sentence = ' '.join(word for word in list_words)
    return sentence

In [8]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def preprocess(text):
    text = cleaningText(text)
    text = casefoldingText(text)
    
    text = text.replace(",", " ")
    
    text = tokenizingText(text)
    text = filteringText(text)
    text = toSentence(text)
    
    text = stemmer.stem(text)
    
    return text

In [14]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

job_texts_clean = job_texts.apply(preprocess)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\andre\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\andre\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

job_vectors = vectorizer.fit_transform(job_texts_clean)

In [16]:
user_input = "python machine learning data"
user_clean = preprocess(user_input)

user_vector = vectorizer.transform([user_clean])

In [17]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

job_vectors = normalize(job_vectors)
user_vector = normalize(user_vector)

similarities = cosine_similarity(user_vector, job_vectors)

In [18]:
import numpy as np

top_n = 5
top_indices = np.argsort(similarities[0])[::-1][:top_n]

recommended_jobs = df.iloc[top_indices]
print(recommended_jobs[['Judul', 'Perusahaan']])

                                               Judul  \
116                                   Data Scientist   
224                                   Data Scientist   
273                                   Data Scientist   
237              Security Endpoint & Data Protection   
234  Logistic Planning Analyst (E-commerce Platform)   

                                     Perusahaan  
116       PT Evolusi Teknologi Solusi (Evoteks)  
224                                        JULO  
273                    PT Global Inovasi Cahaya  
237                PT Clarus Innovace Teknologi  
234  J&T Cargo Indonesia (PT Global Yimi Cargo)  


In [19]:
for i in top_indices:
    print(df.iloc[i]['Perusahaan'], "-> Score:", similarities[0][i])

PT Evolusi Teknologi Solusi (Evoteks) -> Score: 0.7985255109672835
JULO -> Score: 0.5385000309780901
PT Global Inovasi Cahaya -> Score: 0.45026735189202344
PT Clarus Innovace Teknologi -> Score: 0.4499734581602305
J&T Cargo Indonesia (PT Global Yimi Cargo) -> Score: 0.14599665474104417


## dl

In [1]:
import re
import string
import random
import numpy as np
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Layer
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K


In [2]:
df = pd.read_csv("glints_jobs_detail.csv")
job_texts = df['Skills'].astype(str)

In [3]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def cleaningText(text):
    text = re.sub(r"http\S+", '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    text = text.replace('\n', ' ')
    text = text.strip()
    return text

def filteringText(text):
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords.update(stopwords.words('english'))
    
    tokens = word_tokenize(text)
    filtered = [word for word in tokens if word not in listStopwords]
    return " ".join(filtered)

def preprocess(text):
    text = cleaningText(text)
    text = text.replace(",", " ")
    text = filteringText(text)
    text = stemmer.stem(text)
    return text

job_clean = job_texts.apply(preprocess)

In [4]:
pairs_user = []
pairs_job = []
labels = []

job_list = job_clean.tolist()

for i in range(len(job_list)):
    pairs_user.append(job_list[i])
    pairs_job.append(job_list[i])
    labels.append(1)

    rand_idx = random.randint(0, len(job_list)-1)
    pairs_user.append(job_list[i])
    pairs_job.append(job_list[rand_idx])
    labels.append(0)


In [5]:
all_text = pairs_user + pairs_job

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(all_text)

seq_user = tokenizer.texts_to_sequences(pairs_user)
seq_job = tokenizer.texts_to_sequences(pairs_job)

max_len = 20

X_user = pad_sequences(seq_user, maxlen=max_len)
X_job = pad_sequences(seq_job, maxlen=max_len)

y = np.array(labels)

In [6]:
X_user_train, X_user_test, X_job_train, X_job_test, y_train, y_test = train_test_split(
    X_user, X_job, y, test_size=0.2, random_state=42
)

In [7]:
class CosineSimilarityLayer(Layer):
    def call(self, inputs):
        u, v = inputs
        dot = K.sum(u * v, axis=1, keepdims=True)
        norm_u = K.sqrt(K.sum(K.square(u), axis=1, keepdims=True))
        norm_v = K.sqrt(K.sum(K.square(v), axis=1, keepdims=True))
        return dot / (norm_u * norm_v + 1e-8)


In [8]:
vocab_size = len(tokenizer.word_index) + 1

input_user = Input(shape=(max_len,))
input_job = Input(shape=(max_len,))

embedding = Embedding(vocab_size, 128)

user_emb = embedding(input_user)
job_emb = embedding(input_job)

lstm = LSTM(64)

user_vec = lstm(user_emb)
job_vec = lstm(job_emb)

similarity = CosineSimilarityLayer()([user_vec, job_vec])

model = Model(inputs=[input_user, input_job], outputs=similarity)

model.compile(
    loss='mse',
    optimizer='adam',
    metrics=['mae']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 20)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_1 (InputLayer)    │ (None, 20)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 20, 128)           │          57,856 │ input_layer[0][0],         │
│                               │                           │                 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 64)                │          49,408 │ embedding[0][0],           │
│                               │                           │                 │ embedding[1][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ cosine_similarity_layer       │ (None, 1)                 │               0 │ lstm[0][0], lstm[1][0]     │
│ (CosineSimilarityLayer)       │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 107,264 (419.00 KB)

 Trainable params: 107,264 (419.00 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.fit(
    [X_user_train, X_job_train],
    y_train,
    validation_data=([X_user_test, X_job_test], y_test),
    epochs=5,
    batch_size=32
)


Epoch 1/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 83ms/step - loss: 0.0256 - mae: 0.0846 - val_loss: 0.0225 - val_mae: 0.0812
Epoch 2/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0192 - mae: 0.0738 - val_loss: 0.0343 - val_mae: 0.1060
Epoch 3/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.0125 - mae: 0.0577 - val_loss: 0.0323 - val_mae: 0.0996
Epoch 4/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0073 - mae: 0.0412 - val_loss: 0.0322 - val_mae: 0.0990
Epoch 5/5
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0057 - mae: 0.0344 - val_loss: 0.0317 - val_mae: 0.0987


In [29]:
model.save("job_matching_model.keras")

In [10]:
def predict_match(user_text, job_text):
    user_clean = preprocess(user_text)
    job_clean = preprocess(job_text)

    user_seq = tokenizer.texts_to_sequences([user_clean])
    job_seq = tokenizer.texts_to_sequences([job_clean])

    user_pad = pad_sequences(user_seq, maxlen=max_len)
    job_pad = pad_sequences(job_seq, maxlen=max_len)

    score = model.predict([user_pad, job_pad])[0][0]
    return score

user_input = "python react css"

scores = []

for job in job_clean:
    score = predict_match(user_input, job)
    scores.append(score)

top_n = 5
top_indices = np.argsort(scores)[::-1][:top_n]

print("\n=== rekom ===")
for i in top_indices:
    print(df.iloc[i]['Judul'], "-", df.iloc[i]['Perusahaan'], "Score:", scores[i])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 548ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━

In [11]:
model_encoder = Model(inputs=input_user, outputs=user_vec)

In [25]:
user_input = "docker redis golang"
user_seq = tokenizer.texts_to_sequences([user_input])
user_pad = pad_sequences(user_seq, maxlen=max_len)


In [26]:
all_job_pads = pad_sequences(tokenizer.texts_to_sequences(job_clean), maxlen=max_len)
job_vectors = model_encoder.predict(all_job_pads)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [27]:
user_vector = model_encoder.predict(user_pad)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step


In [28]:
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(user_vector, job_vectors)[0]

In [29]:
top_indices = np.argsort(scores)[::-1][:5] 

print("\n=== top 5 ===")
for i in top_indices:
    judul = df.iloc[i]['Judul']
    perusahaan = df.iloc[i]['Perusahaan']
    sekil = df.iloc[i]['Skills']
    skor_kemiripan = scores[i]
    
    print(f"Hasil: {judul} di {perusahaan}")
    print(f"skilnya {sekil}")
    print(f"Skor Kemiripan: {skor_kemiripan:.4f}")
    print("-" * 30)


=== top 5 ===
Hasil: Backend Golang Developer di PT Entrepreneur Trust Digital
skilnya Docker, Redis, Go / Golang
Skor Kemiripan: 0.9485
------------------------------
Hasil: Golang Backend Developer (Hybrid) di Pt. Empore Hezer Tama
skilnya GIT, SQL, Restful API, English Language, Go / Golang
Skor Kemiripan: 0.8050
------------------------------
Hasil: Software Developer di PT Sigma Global Teknologi
skilnya Spring Boot, Java, Rust, Go / Golang
Skor Kemiripan: 0.7782
------------------------------
Hasil: Backend Developer di PT. ASABA DIGITAL INNOTECH
skilnya MySQL, Go / Golang, PostgreSQL
Skor Kemiripan: 0.7480
------------------------------
Hasil: Golang Developer - Banking di PT Sigma Global Teknologi
skilnya Gin, Go / Golang, PostgreSQL
Skor Kemiripan: 0.7447
------------------------------
